# Imports


In [1]:
from __future__ import annotations

import gzip
import math
import struct
from pathlib import Path
from typing import Iterable

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split


# Constants


In [2]:
RANDOM_SEED = 42


In [3]:
DATA_DIR: Path = Path("/home/linkezio/Datasets/MNIST")


In [4]:
MODELS_DIR: Path = Path("/home/linkezio/Projects/Efficient-Polling-Based-Learning-Rate-Optimization-for-Neural-Networks/models")


In [5]:
EPOCHS = 20

# Configs


## Seeds


In [6]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## Device


In [7]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


device: cuda


# Data


## IDX helpers


In [8]:
# Opens a regular or gzipped binary file.
def _open_maybe_gzip(path: Path):
    return gzip.open(path, "rb") if path.suffix == ".gz" else path.open("rb")


# Reads an IDX/ubyte file and returns data in its original shape.
def _read_idx(path: Path) -> np.ndarray:
    with _open_maybe_gzip(path) as f:
        header = f.read(4)
        if len(header) != 4:
            raise ValueError(f"Invalid IDX file: incomplete header in {path}")

        zero_1, zero_2, data_type, dims = struct.unpack(">BBBB", header)
        if (zero_1, zero_2) != (0, 0):
            raise ValueError(f"Invalid IDX file: incorrect prefix in {path}")
        if data_type != 0x08:
            raise ValueError(f"Unsupported IDX type ({data_type}) em {path}")

        shape = tuple(struct.unpack(">I", f.read(4))[0] for _ in range(dims))
        data = f.read()

    arr = np.frombuffer(data, dtype=np.uint8)
    expected_size = int(np.prod(shape))
    if arr.size != expected_size:
        raise ValueError(
            f"Inconsistent IDX in {path}: expected {expected_size} elements, got {arr.size}"
        )
    return arr.reshape(shape)


# Automatically locates MNIST train/test files.
def find_mnist_files(data_dir: Path) -> dict[str, Path]:
    candidates = [p for p in data_dir.glob("*") if p.is_file()]
    names = {p.name: p for p in candidates}

    def pick(prefixes: Iterable[str]) -> Path:
        for p in candidates:
            lower = p.name.lower()
            if any(lower.startswith(pref) for pref in prefixes) and ("idx" in lower or "ubyte" in lower):
                return p
        raise FileNotFoundError(f"Could not find MNIST files in {data_dir}. Files found: {sorted(names)}")

    return {
        "train_images": pick(["train-images", "train_images", "train-images-idx3"]),
        "train_labels": pick(["train-labels", "train_labels", "train-labels-idx1"]),
        "test_images": pick(["t10k-images", "test-images", "t10k_images", "t10k-images-idx3"]),
        "test_labels": pick(["t10k-labels", "test-labels", "t10k_labels", "t10k-labels-idx1"]),
    }


## Dataset Class


In [9]:
class MNISTDataset(Dataset):
    def __init__(
        self,
        data_dir: Path,
        train: bool,
        mean: torch.Tensor | None = None,
        std: torch.Tensor | None = None,
    ):
        files = find_mnist_files(data_dir)
        if train:
            images_path, labels_path = files["train_images"], files["train_labels"]
        else:
            images_path, labels_path = files["test_images"], files["test_labels"]

        images = _read_idx(images_path)
        labels = _read_idx(labels_path)
        if images.ndim != 3:
            raise ValueError(f"Expected MNIST images (N,H,W). Received: {images.shape}")
        if labels.ndim != 1:
            raise ValueError(f"Labels MNIST expected (N,). Received: {labels.shape}")
        if images.shape[0] != labels.shape[0]:
            raise ValueError("Number of images != number of labels")

        self.images = torch.from_numpy(images).float().unsqueeze(1)  # (N,1,28,28), 0–255
        self.labels = torch.from_numpy(labels).long()

        if (mean is None) ^ (std is None):
            raise ValueError("Pass `mean` and `std` together, or neither.")
        self.mean = mean
        self.std = std

    # Returns the total number of dataset samples.
    def __len__(self) -> int:
        return int(self.labels.shape[0])

    # Returns one sample (x, y), with optional normalization.
    def __getitem__(self, idx: int):
        x = self.images[idx]
        if self.mean is not None:
            x = (x - self.mean) / self.std
        y = self.labels[idx]
        return x, y


## Calculate mean and std for normalize later


In [10]:

# Computes per-channel mean and standard deviation over the full dataset.
def compute_mean_std(dataset, batch_size=512):
    loader_mean = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    channel_sum = None
    n_pixels = 0
    for images, _ in loader_mean:
        b, c, h, w = images.shape
        if channel_sum is None:
            channel_sum = torch.zeros(c, dtype=torch.float64)
        channel_sum += images.double().sum(dim=(0, 2, 3))
        n_pixels += b * h * w

    mean = (channel_sum / n_pixels).float()

    loader_var = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    sum_sq = None
    for images, _ in loader_var:
        b, c, h, w = images.shape
        if sum_sq is None:
            sum_sq = torch.zeros(c, dtype=torch.float64)
        diff = images.double() - mean.view(1, c, 1, 1).double()
        sum_sq += (diff * diff).sum(dim=(0, 2, 3))

    var = (sum_sq / n_pixels).float()
    std = torch.sqrt(var)
    std = torch.clamp(std, min=1e-8)
    return mean, std


In [11]:
train_for_stats = MNISTDataset(DATA_DIR, train=True)

mnist_mean, mnist_std = compute_mean_std(train_for_stats)

mnist_mean = mnist_mean.view(1, 1, 1)
mnist_std = mnist_std.view(1, 1, 1)

print("mean (1 channel):", mnist_mean.squeeze().tolist())
print("std  (1 channel):", mnist_std.squeeze().tolist())


/tmp/ipykernel_106897/3834158628.py:24: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.images = torch.from_numpy(images).float().unsqueeze(1)  # (N,1,28,28), 0–255


mean (1 channel): 33.31842041015625
std  (1 channel): 78.56748962402344


## Train, Validation, Test Split


In [12]:
class DataLoaderHyperparameters:
    batch_size: int = 128
    val_fraction: float = 0.1
    num_workers: int = 0  # Jupyter: use 0 (workers cannot resolve classes in __main__).

data_loader_hyperparameters = DataLoaderHyperparameters()


In [13]:
full_train = MNISTDataset(DATA_DIR, train=True, mean=mnist_mean, std=mnist_std)
test_ds = MNISTDataset(DATA_DIR, train=False, mean=mnist_mean, std=mnist_std)

val_size = max(1, int(len(full_train) * data_loader_hyperparameters.val_fraction))
train_size = len(full_train) - val_size

train_ds, val_ds = random_split(
    full_train,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

train_loader = DataLoader(
    train_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=True,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
val_loader = DataLoader(
    val_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
test_loader = DataLoader(
    test_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)

len(train_ds), len(val_ds), len(test_ds)


(54000, 6000, 10000)

# Model


## Hyperparameters


In [14]:
class ModelHyperparameters:
    batch_size: int = 128
    epochs: int = EPOCHS
    lr: float = 1e-3
    weight_decay: float = 0.0
    num_workers: int = 0  # Jupyter

model_hyperparameters = ModelHyperparameters()


## Model Class


In [15]:
class SimpleMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    # Runs the model forward pass.
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)


# Training


## Metric Functions


In [16]:
# Computes average batch accuracy.
def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


In [17]:
loss_fn = nn.CrossEntropyLoss()


## Epoch Functions


In [18]:
@torch.inference_mode()
# Evaluates the model for one epoch and returns average loss/accuracy.
def eval_epoch(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[float, float]:
    model.eval()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        losses.append(loss_fn(logits, y).item())
        accs.append(accuracy(logits, y))
    return float(np.mean(losses)), float(np.mean(accs))


# Trains the model for one epoch and returns average loss/accuracy.
def train_epoch(model: nn.Module, loader: DataLoader, optim: torch.optim.Optimizer, loss_fn: nn.Module) -> tuple[float, float]:
    model.train()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optim.step()

        losses.append(loss.item())
        accs.append(accuracy(logits.detach(), y))
    return float(np.mean(losses)), float(np.mean(accs))


In [19]:
model = SimpleMNISTCNN().to(DEVICE)
optim = torch.optim.Adam(
    model.parameters(),
    lr=model_hyperparameters.lr,
    weight_decay=model_hyperparameters.weight_decay,
)


In [20]:
# Coordinates epoch training/validation and saves the best checkpoint.
def fit_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    epochs: int,
    model_path: Path,
) -> tuple[dict[str, list[float]], float, Path]:
    """Coordinates epoch training/validation and saves the best checkpoint."""
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    best_val_acc = -math.inf

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optim, loss_fn)
        va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), model_path)

        print(
            f"epoch {epoch:02d}/{epochs} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print("best val acc:", best_val_acc, "saved:", str(model_path))
    return history, best_val_acc, model_path


In [ ]:
history, best_val_acc, model_path = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim,
    loss_fn=loss_fn,
    epochs=model_hyperparameters.epochs,
    model_path=MODELS_DIR / "mnist_best.pt",
)

epoch 01/20 | train loss 0.1899 acc 0.9425 | val loss 0.0657 acc 0.9816
epoch 02/20 | train loss 0.0490 acc 0.9848 | val loss 0.0545 acc 0.9839
epoch 03/20 | train loss 0.0322 acc 0.9898 | val loss 0.0509 acc 0.9868
epoch 04/20 | train loss 0.0245 acc 0.9920 | val loss 0.0480 acc 0.9852
epoch 05/20 | train loss 0.0183 acc 0.9939 | val loss 0.0434 acc 0.9887


# Test


In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / "mnist_best.pt", map_location=DEVICE))

test_loss, test_acc = eval_epoch(model, test_loader, loss_fn)

print(f"test loss {test_loss:.4f} | test acc {test_acc:.4f}")
